# ARC-AGI-2 × Gemini — program search

Gemini writes a Python `transform()` for each ARC-AGI-2 task. Each program is run on the train pairs and kept only if it gets all of them exactly right. Results are saved to Google Drive and pushed to GitHub.

**Not for Kaggle submission:** ARC Prize evaluations run without internet and forbid API-based systems. Use the verified programs as ideas for the offline DSL solver (`ARC-AGI-2/pipeline.py`).

### One-time setup
1. In the left sidebar, open **Secrets** (key icon) and add:
   - `GEMINI_API_KEY`: your Gemini API key
   - `GITHUB_TOKEN`: a fine-grained GitHub token with *Contents: read and write* on `algilani2008/Kaggle-Quantum-AI` only
2. Turn on **Notebook access** for both secrets.

Then **Runtime → Run all**. Each run picks up where the last one stopped.

In [ ]:
# ---- Settings ---------------------------------------------------------------
MODEL      = "gemini-3.8-flash" # cheaper; use "gemini-4-argon" for the strongest (and priciest) model
THINKING   = "high"             # low | medium | high
MAX_TASKS  = 20                 # new tasks per run (cost control)
MAX_CALLS  = 200                # hard cap on API calls per run
SAMPLES    = 2                  # independent attempts per task
ROUNDS     = 2                  # feedback rounds per attempt
SPLIT      = "evaluation"       # evaluation (120 tasks, scored) or training (1000 tasks)

REPO       = "algilani2008/Kaggle-Quantum-AI"
DRIVE_DIR  = "/content/drive/MyDrive/Kaggle_Projects/Kaggle_Quantum_Data/ARC-AGI-2_gemini"

In [ ]:
# ---- Secrets (never printed) ------------------------------------------------
import os, base64
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
# Git auth header kept in memory only; nothing is written to .git/config.
GIT_AUTH = "Authorization: Basic " + base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
print("Secrets loaded.")

In [ ]:
# ---- Google Drive -----------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive folder:", DRIVE_DIR)

In [ ]:
# ---- Code and data ----------------------------------------------------------
import subprocess, shutil

def git(*args, cwd=None):
    r = subprocess.run(["git", "-c", f"http.extraHeader={GIT_AUTH}", *args],
                       cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        # Never echo the command line: it contains the auth header.
        raise RuntimeError(f"git {args[0]} failed: {r.stderr.strip()[-500:]}")
    return r.stdout

shutil.rmtree("/content/repo", ignore_errors=True)
git("clone", "--depth", "1", f"https://github.com/{REPO}", "/content/repo")
if not os.path.exists("/content/arc_data"):
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/arcprize/ARC-AGI-2", "/content/arc_data"], check=True)
subprocess.run(["pip", "-q", "install", "requests", "numpy"], check=True)
print("Repo and ARC-AGI-2 data ready.")

In [ ]:
# ---- Resume from Drive ------------------------------------------------------
OUT = "/content/gemini_out"
shutil.rmtree(OUT, ignore_errors=True)
if os.path.exists(os.path.join(DRIVE_DIR, "state.json")):
    shutil.copytree(DRIVE_DIR, OUT, ignore=shutil.ignore_patterns("reports"))
    print("Resumed previous progress from Drive.")
else:
    os.makedirs(OUT)
    print("First run: starting fresh.")

In [ ]:
# ---- Run the search ---------------------------------------------------------
!python /content/repo/ARC-AGI-2/gemini/program_search.py \
    --tasks-dir /content/arc_data/data/{SPLIT} --out-dir {OUT} \
    --model {MODEL} --thinking {THINKING} --max-tasks {MAX_TASKS} \
    --max-calls {MAX_CALLS} --samples {SAMPLES} --rounds {ROUNDS}

In [ ]:
# ---- Save to Drive ----------------------------------------------------------
from datetime import datetime
stamp = datetime.now().strftime("%Y-%m-%d_%H%M")
for name in ("state.json", "report.json", "report.md"):
    shutil.copy(os.path.join(OUT, name), DRIVE_DIR)
shutil.copytree(os.path.join(OUT, "programs"), os.path.join(DRIVE_DIR, "programs"), dirs_exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, "reports"), exist_ok=True)
shutil.copy(os.path.join(OUT, "report.md"), os.path.join(DRIVE_DIR, "reports", f"report_{stamp}.md"))
print("Saved to", DRIVE_DIR)

In [ ]:
# ---- Push to GitHub ---------------------------------------------------------
dest = "/content/repo/ARC-AGI-2/gemini/results"
os.makedirs(dest, exist_ok=True)
shutil.copytree(os.path.join(OUT, "programs"), os.path.join(dest, "programs"), dirs_exist_ok=True)
for name in ("state.json", "report.md"):
    shutil.copy(os.path.join(OUT, name), dest)

repo = "/content/repo"
subprocess.run(["git", "config", "user.name", "ARC Gemini Colab"], cwd=repo, check=True)
subprocess.run(["git", "config", "user.email", "noreply@users.noreply.github.com"], cwd=repo, check=True)
subprocess.run(["git", "add", "ARC-AGI-2/gemini/results"], cwd=repo, check=True)
if subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=repo).returncode == 0:
    print("Nothing new to push.")
else:
    subprocess.run(["git", "commit", "-q", "-m", f"Gemini program search results {stamp}"], cwd=repo, check=True)
    try:
        git("push", "origin", "HEAD:main", cwd=repo)
    except RuntimeError:
        git("pull", "--rebase", "origin", "main", cwd=repo)
        git("push", "origin", "HEAD:main", cwd=repo)
    sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, capture_output=True, text=True).stdout.strip()
    print(f"Pushed: https://github.com/{REPO}/commit/{sha}")

In [ ]:
# ---- Summary ----------------------------------------------------------------
print(open(os.path.join(OUT, "report.md")).read())